# Week 1 — Foundations and Financial Data

FIN 43800 · Machine Learning in Finance

Andreas Neuhierl  
2026-10-21

# What is Machine Learning?

## Two Cultures

Leo Breiman, *Statistical Modeling: The Two Cultures* (2001):

- **Data modeling** — assume a stochastic model, estimate its
  parameters. Goal: describe *nature’s mechanism*.
- **Algorithmic modeling** — the mechanism is unknown; judge a model
  only by how well it predicts. Goal: *accuracy*.

## The Black Box

Data comes out of a box. You do not get to see inside.

- **Data modeling** fills the box with an assumed equation, then
  estimates it.
- **Algorithmic modeling** leaves the box shut and builds something
  outside it that predicts well.

The second one is what “machine learning” usually means.

## Which One Is This Course?

**Both, and you must know which you are doing.**

- “What is Apple’s beta?” — explanation. You want the parameter itself.
- “What will the market return next month?” — prediction. Nobody cares
  what the coefficients are if the forecast is good.

This course is mostly the second. The organizing question for fourteen
meetings: **are you explaining, or predicting?**

# What You Already Know

## Assumed: CAPM and Portfolio Basics

This course assumes you have met these. We are not re-teaching them.

- **Diversification** — combining imperfectly correlated assets lowers
  portfolio variance without lowering expected return proportionally.
- **Systematic vs. idiosyncratic risk** — only the part you cannot
  diversify away should earn a premium.
- **CAPM** — an asset’s expected excess return is proportional to its
  beta: $E[r_i - r_f] = \beta_i \times E[r_m - r_f]$.
- **Beta** — the slope from regressing an asset’s excess return on the
  market’s.

If any of that is unfamiliar, review this ASAP!

# What a Statistical Model Is

## The Market Model

$$ r_i = \alpha + \beta \, r_m + \varepsilon $$

Both returns are **excess** returns — already net of the risk-free rate.

|  |  |  |
|------------------------|------------------------|------------------------|
| **Model** | $r_i = \alpha + \beta r_m + \varepsilon$ | the relationship we assume |
| **Parameters** | $\alpha, \beta$ | unknown, fixed, never observed |
| **Estimates** | $\widehat\alpha, \widehat\beta$ | computed from a sample |
| **Fitted values** | $\widehat r_i = \widehat\alpha + \widehat\beta r_m$ | what the model says |
| **Residuals** | $r_i - \widehat r_i$ | what it missed |

## Three Words for the Same Thing

$r_m$ is a **predictor**. It is also a **feature**. It is also a
**regressor**.

- **Regressor** — econometrics. You will see it in the finance
  literature.
- **Predictor** — statistics. Used throughout ESL and ISLP.
- **Feature** — machine learning. Used in scikit-learn’s documentation.

Same object, three communities. This course uses all three,
deliberately, because you will meet all three.

## Where the Code Runs: Google Colab

**github.com/aneuhierl/ml-in-finance-fall2026** — every notebook, with
an “Open in Colab” link. The same link is on Brightspace. This week:
[open the week 1 notebook in
Colab](https://colab.research.google.com/github/aneuhierl/ml-in-finance-fall2026/blob/main/notebooks/w01-foundations.ipynb).

- **You need:** a **personal Google account** (Gmail) — not your Purdue
  login — and a laptop with a current Chrome, Firefox or Safari. Nothing
  to install; it is free.
- **Once per notebook:** File → **Save a copy in Drive**, then work in
  your copy. Edits to the original are not saved.
- **Every session:** download the data file from **Brightspace**, click
  the folder icon on Colab’s left edge, and drag the file in. Colab
  deletes it when the session ends.

## Optional: Python on Your Own Laptop

Colab is all this course requires. If you would rather work offline:

- **Install Anaconda** — free for coursework:
  [anaconda.com/download](https://www.anaconda.com/download) (“Skip
  registration” is fine). Needs about 5 GB of disk.
  - **Windows 11:** [install
    guide](https://www.anaconda.com/docs/getting-started/anaconda/install/windows-gui-install).
    Choose “Just Me”; leave “Add to PATH” unchecked.
  - **Mac:** [install
    guide](https://www.anaconda.com/docs/getting-started/anaconda/install/mac-gui-install).
    The current installer is for Apple-silicon Macs only.
- **Get the notebooks:** on the course GitHub page, Code → Download ZIP,
  and unzip it. Put Brightspace data files in a folder named `data` next
  to the notebook.
- **Open a notebook** from JupyterLab, launched from Anaconda Navigator.
- **One extra package:** run `pip install yfinance` once. (Later, for
  the text week: `conda install gensim`, not pip.)

## Two Return Conventions

Keep both straight, all semester:

- **Simple** returns, $P_t/P_{t-1} - 1$ — not **log** returns,
  $\ln(P_t/P_{t-1})$.
- **Excess** returns, net of the risk-free rate — not **gross** returns,
  with the risk-free rate still included.

The panel we load next is **simulated** — one market factor plus 25
assets — so we know the truth it came from and can check whether our
methods recover it.

## Setup

Run this cell first — after dragging `returns.csv` from Brightspace into
Colab.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Fetch the one helper file we need from the course repo.
    import subprocess
    subprocess.run(
        ["wget", "-q", "-O", "mlfin_data.py",
         "https://raw.githubusercontent.com/aneuhierl/ml-in-finance-fall2026/main/src/mlfin_data.py"],
        check=True,
    )
else:
    # Find src/mlfin_data.py by walking UP from this notebook's folder, so the
    # import works wherever the notebook was saved (fresh clone: ../src).
    here = Path.cwd().resolve()
    src_folders = [folder / "src" for folder in [here, *here.parents]]
    matches = [folder for folder in src_folders if (folder / "mlfin_data.py").is_file()]
    sys.path.insert(0, str(matches[0]) if matches else "../src")

from mlfin_data import load_returns

# Simple monthly EXCESS returns in decimal form: 0.01 means one percent.
returns = load_returns()
print(returns.shape)
returns.iloc[:3, :4]

## Annualizing, and why it is not one rule

- Mean scales linearly with time
- Standard deviation scales with its *square root* — one multiplier does
  not fit both.
- Both are not w/out assumptions

In [ ]:
# Mean scales with time; standard deviation with its square root (above).
# Both also approximate true multiplicative compounding -- fine here
# because monthly returns are small.
monthly_mean = returns["market"].mean()
monthly_std = returns["market"].std()

annual_mean = monthly_mean * 12
annual_std = monthly_std * (12 ** 0.5)

print(f"annualized mean:   {annual_mean:.3f}")
print(f"annualized stdev:  {annual_std:.3f}")
print(f"Sharpe ratio:      {annual_mean / annual_std:.3f}")

## Our first model: one asset on the market

In [ ]:
from sklearn.linear_model import LinearRegression

# X must be 2-dimensional for sklearn, even with a single predictor.
X = returns[["market"]].values
y = returns["asset_01"].values

model = LinearRegression().fit(X, y)

print(f"beta:  {model.coef_[0]:.3f}")
print(f"alpha: {model.intercept_:.5f}")
print(f"R^2:   {model.score(X, y):.3f}")

## The picture

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(9, 5))
axis.scatter(returns["market"], returns["asset_01"], s=12, alpha=0.5)

line_x = [returns["market"].min(), returns["market"].max()]
line_y = [model.predict([[x]])[0] for x in line_x]
axis.plot(line_x, line_y, linewidth=2)

axis.set_xlabel("Market excess return (monthly)")
axis.set_ylabel("Asset 01 excess return (monthly)")
axis.set_title(f"Beta = {model.coef_[0]:.2f}")
plt.show()

# “Estimation Is Optimization”

## Choosing $\widehat\beta$

We have a model. How do we pick the numbers?

**Least squares:** pick the $\widehat\beta$ that makes the residuals
smallest.

$$ \widehat\beta = \arg\min_{b} \sum_{t} (r_{i,t} - a - b \, r_{m,t})^2 $$

- $\arg\min$ is doing all the work-
- Estimating a model means **solving an optimization problem**
- Python does it for us.

## Seeing the Minimum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

market = returns["market"].values
asset = returns["asset_01"].values
intercept = model.intercept_  # pinned at sklearn's own fit, so we can plot in 1-D

# Try many candidate slopes, in steps of 0.001, and record how badly each fits.
beta_grid = np.linspace(-0.5, 2.5, 3001)
sum_of_squares = [
    float(np.sum((asset - intercept - candidate * market) ** 2))
    for candidate in beta_grid
]

best = beta_grid[int(np.argmin(sum_of_squares))]

figure, axis = plt.subplots(figsize=(9, 5))
axis.plot(beta_grid, sum_of_squares, linewidth=2)
axis.axvline(best, linestyle="--")
axis.set_xlabel("candidate slope $b$", fontsize=13)
axis.set_ylabel("sum of squared residuals", fontsize=13)
axis.set_title(f"minimized at b = {best:.3f}", fontsize=15)
axis.tick_params(labelsize=11)
plt.show()

## Getting Up To Speed

**No prior Python is assumed.**

- **ISLP — *An Introduction to Statistical Learning with Python*.** Free
  PDF: <https://www.statlearning.com/resources-python>. This is the book
  to read.
- **The ISLP labs**, runnable chapter by chapter:
  <https://github.com/intro-stat-learning/ISLP_labs/tree/stable>

<!-- -->

- **DataCamp**, free through Purdue. Optional, but the fastest way in if
  you have never written a line of Python — the invite link is on
  Brightspace; join with your @purdue.edu address.

<!-- -->

- **And AI.** See the policy: you are expected to use it.

# In Sample and Out of Sample

## The Only Question That Matters

- Fit on data you have. Judge on data you did **not**.
- This is the criterion for **prediction**. For explanation — “what *is*
  Apple’s beta?” — the in-sample estimate itself is the object of
  interest.

In [ ]:
from sklearn.linear_model import LinearRegression

half = len(returns) // 2
X = returns[["market"]].values
y = returns["asset_01"].values

early = LinearRegression().fit(X[:half], y[:half])

print(f"in-sample  R^2, first {half} months: {early.score(X[:half], y[:half]):.3f}")
print(f"out-of-sample R^2, last {len(returns) - half} months: {early.score(X[half:], y[half:]):.3f}")

## In-Sample $R^2$ Is Not the Criterion

In [ ]:
import numpy as np

generator = np.random.default_rng(0)

# Add predictors that are PURE NOISE -- they cannot possibly help.
for how_many in (0, 5, 20, 50):
    noise = [generator.normal(size=len(X)) for _ in range(how_many)]
    X_padded = np.column_stack([X] + noise) if noise else X
    fit = LinearRegression().fit(X_padded, y)
    print(f"{how_many:2d} random predictors -> in-sample R^2 {fit.score(X_padded, y):.3f}")

- In-sample $R^2$ **never goes down** when you add a predictor. Not even
  a useless one.
- Modern models often have **very many** features — hundreds of firm
  characteristics, thousands of words. In-sample fit is then
  meaningless.

# The Curse of Dimensionality

## The Same Data, Spread Thinner

Twenty points, spread uniformly over $[0, 2]$ in each dimension. How
many land in the unit bin $[0, 1]$?

In [ ]:
import numpy as np

# Seed 23 reproduces the counts in Parsons, Haque & Liu (2004), Figure 1.
generator = np.random.default_rng(23)
points = generator.uniform(0, 2, size=(20, 3))


def count_in_unit_bin(points, n_dimensions):
    """How many points have every one of their first n coordinates in [0, 1]?"""
    count = 0
    for point in points:
        if max(point[:n_dimensions]) <= 1.0:
            count += 1
    return count


for n_dimensions in (1, 2, 3):
    count = count_in_unit_bin(points, n_dimensions)
    print(f"{n_dimensions} dimension(s): {count} of 20 points in the unit bin")

## Same Points, Pictured

Same 20 points. Blue ones sit inside the unit bin; gray ones do not.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 -- registers the 3-D projection

figure = plt.figure(figsize=(11, 5.5))

# Panel 1: one dimension -- the bin is the segment [0, 1] on a line.
axis_1 = figure.add_subplot(1, 3, 1)
for x, y, z in points:
    color = "tab:blue" if x <= 1.0 else "tab:gray"
    axis_1.scatter(x, 0, s=140, color=color)
axis_1.axvspan(0, 1, color="tab:orange", alpha=0.25)
axis_1.set_xlim(0, 2)
axis_1.set_yticks([])
axis_1.set_xlabel("dimension 1", fontsize=16)
axis_1.set_title(f"p = 1: {count_in_unit_bin(points, 1)} of 20 in bin", fontsize=20)
axis_1.tick_params(labelsize=14)

# Panel 2: two dimensions -- the bin is the unit square.
axis_2 = figure.add_subplot(1, 3, 2)
for x, y, z in points:
    inside = (x <= 1.0) and (y <= 1.0)
    color = "tab:blue" if inside else "tab:gray"
    axis_2.scatter(x, y, s=140, color=color)
axis_2.add_patch(plt.Rectangle((0, 0), 1, 1, color="tab:orange", alpha=0.25))
axis_2.set_xlim(0, 2)
axis_2.set_ylim(0, 2)
axis_2.set_xlabel("dimension 1", fontsize=16)
axis_2.set_ylabel("dimension 2", fontsize=16)
axis_2.set_title(f"p = 2: {count_in_unit_bin(points, 2)} of 20 in bin", fontsize=20)
axis_2.tick_params(labelsize=14)

# Panel 3: three dimensions -- the bin is the unit cube.
axis_3 = figure.add_subplot(1, 3, 3, projection="3d")
for x, y, z in points:
    inside = (x <= 1.0) and (y <= 1.0) and (z <= 1.0)
    color = "tab:blue" if inside else "tab:gray"
    axis_3.scatter(x, y, z, s=70, color=color)

# Draw the unit cube's 12 edges to mark the bin.
bottom_square = [(0, 0, 0), (1, 0, 0), (1, 1, 0), (0, 1, 0), (0, 0, 0)]
top_square = [(0, 0, 1), (1, 0, 1), (1, 1, 1), (0, 1, 1), (0, 0, 1)]
for square in (bottom_square, top_square):
    square_x = [corner[0] for corner in square]
    square_y = [corner[1] for corner in square]
    square_z = [corner[2] for corner in square]
    axis_3.plot(square_x, square_y, square_z, color="tab:orange", alpha=0.6)
for x, y in ((0, 0), (1, 0), (1, 1), (0, 1)):
    axis_3.plot([x, x], [y, y], [0, 1], color="tab:orange", alpha=0.6)

axis_3.set_xlim(0, 2)
axis_3.set_ylim(0, 2)
axis_3.set_zlim(0, 2)
axis_3.set_xlabel("dim 1", fontsize=16, labelpad=12)
axis_3.set_ylabel("dim 2", fontsize=16, labelpad=12)
axis_3.set_zlabel("dim 3", fontsize=16, labelpad=8)
axis_3.set_title(f"p = 3: {count_in_unit_bin(points, 3)} of 20 in bin", fontsize=20)
axis_3.tick_params(labelsize=14)

plt.tight_layout()
plt.show()

## But Breiman Said Dimensionality Is a *Blessing*

He did. Section 10 of the same paper: *“Instead of reducing
dimensionality, increase it by adding many functions of the predictor
variables.”*

Both are true, and the difference is the whole course:

- The **curse** bites *local averaging*. Neighbors stop being near.
- The **blessing** is real for methods that do **not** average locally —
  trees and forests partition adaptively; support vector machines use
  margins.
- Those methods extract small pieces of information from many weak
  predictors.
- So the curse is not “avoid many predictors”. **It is the reason you
  need the methods we spend fourteen meetings on.**

# Where This Course Goes

## Almost Everything Here Is a Time Series

A **time series** is a variable observed repeatedly over time, in order,
where the order matters. `returns["market"]` is 360 monthly
observations, oldest first.

Order is not decoration:

- You may only use the past to predict the future.
- Observations near each other in time are related in ways that random
  samples are not.
- Shuffling the rows destroys the thing you are trying to model.

## Two Axes, Fourteen Meetings

**How many series?**

- *Now:* one at a time. Predict the market’s return next month.
- *Later:* thousands at once — every stock in the cross-section,
  together.

**How many predictors?**

- *Now:* one. `r_m`.
- *Later:* hundreds of firm characteristics.

Machine learning earns its keep in the bottom-right corner: **many
series, many predictors**. That is also exactly where the curse would
kill local averaging — which is why the methods matter.

## Before Monday

1.  Open the week 1 notebook in Colab, save a copy in Drive, drag in
    `returns.csv` from Brightspace, and run the first cell.
2.  If anything breaks, paste the error into an AI assistant first —
    that is what it is for. The TA is the last resort, not the first.
3.  **Optional, and worth it:** estimate the beta of every asset in the
    panel and find the three highest. A loop is fine; clarity beats
    cleverness.

## Readings

**Required** (on Brightspace): Breiman (2001), *Statistical Modeling:
The Two Cultures*, Statistical Science 16(3).

**Optional:**

- Donoho (2000), *High-Dimensional Data Analysis: The Curses and
  Blessings of Dimensionality*.
- Parsons, Haque & Liu (2004), *Subspace Clustering for High Dimensional
  Data*, SIGKDD Explorations 6(1) — the source of today’s bin figure.
- Peyré, Delon & Fadili, *Curses and Blessings of High Dimension*:
  <https://mathematical-coffees.github.io/mc08-high-dimension/>
- ISLP, chapter 2.